## Lastspitzenmanagement mit Elektrodenkessel & Batteriespeicher

Ein bestehender elektrischer **Lastbedarf (Grundlast)** einer Kundenanlage soll durch ein
zusätzliches Modul – einen **Elektrodenkessel (EK)** zur Wärmeerzeugung – erweitert werden.
Der Kessel erhöht die elektrische Last und kann die vorhandene **Netzanschlussleistung**
überschreiten.

Ein Überschreiten der Netzanschlussleistung ist **nur zu bestimmten, frei wählbaren
Freigabe-Stunden** erlaubt (z. B. nachts). Außerhalb dieser Zeiten muss der Netzbezug unter der
regulären Anschlussleistung bleiben. Um den Kessel dennoch jederzeit betreiben zu können, wird
zusätzlich ein **Batteriespeicher** installiert, der Leistungsspitzen glättet (Peak-Shaving).

**Aufgabe der Optimierung:** die **minimal erforderliche Batteriekapazität** bestimmen, mit der
Grundlast + Elektrodenkessel gedeckt werden können, ohne die Netzanschlussleistung außerhalb der
Freigabe-Stunden zu überschreiten.

**Ablauf des Notebooks**
1. Bestehenden Lastbedarf (Grundlast) und Wärmebedarf erzeugen.
2. Im Konfigurations-Fenster die neuen Module & Netzgrenzen eintragen (Dummy-Werte).
3. Optimierungsmodell (Pyomo) aufstellen und lösen → Batteriegröße wird dimensioniert.
4. Ergebnisse in mehreren eigenständigen Diagrammen darstellen (folientauglich für PowerPoint).


### 1 · Bibliotheken importieren

In [189]:
import numpy as np
import pandas as pd
import pyomo.environ as pyoe
from pyomo.opt import SolverFactory
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# ---- einheitliches, technisches Layout (Arial) für alle Plots -----------
FARBEN = dict(grund="#9ecae1", kessel="#08519c", laden="#2ca02c", entladen="#c00000",
              netz="#111111", soc="#7d3ac1", limit="#c00000", freigabe="#f6a600")

# professionelles Basistemplate: Arial, Achsenlinien mit Rahmen, dezentes Gitter
pio.templates.default = "plotly_white"
_tpl = pio.templates["plotly_white"]
_tpl.layout.font.update(family="Arial, Helvetica, sans-serif", size=14, color="#222222")
_tpl.layout.title.font.update(family="Arial, Helvetica, sans-serif", size=18, color="#111111")
_tpl.layout.paper_bgcolor = "white"
_tpl.layout.plot_bgcolor  = "white"
for _ax in (_tpl.layout.xaxis, _tpl.layout.yaxis):
    _ax.update(showline=True, linecolor="#444444", linewidth=1, mirror=True,
               ticks="outside", tickcolor="#444444", ticklen=5,
               gridcolor="rgba(0,0,0,0.09)", zeroline=False, title_font=dict(size=15))

# Export-Konfiguration: PNG in 3-facher Auflösung (für PowerPoint) über die Kamera-Schaltfläche
PLOT_CONFIG = {"toImageButtonOptions": {"format": "png", "scale": 3}, "displaylogo": False}

def basis_layout(fig, titel, ytitel="Leistung [MW]", h=520, zeit_achse=True):
    # Einheitliches Layout: Titel oben verankert, Legende als eigenes Band, technische Achsen
    fig.update_layout(title=dict(text=titel, x=0.0, xanchor="left", y=0.97, yanchor="top"),
                      xaxis_title="Zeit", yaxis_title=ytitel, height=h, width=1000,
                      legend=dict(orientation="h", yanchor="bottom", y=1.03, x=0,
                                  bgcolor="rgba(255,255,255,0)"),
                      margin=dict(t=120, r=30, l=75, b=60))
    if zeit_achse:                                  # Zeitachse technisch: nur HH:MM, 3-h-Raster
        fig.update_xaxes(tickformat="%H:%M", dtick=3 * 3600 * 1000)
    return fig

### 2 · Bestehender Lastbedarf & Wärmebedarf (Dummy, mehrfach variierend)

Synthetische Tagesprofile in 15-Minuten-Auflösung mit mehreren Tagesspitzen und leichtem
Messrauschen (fester Zufalls-Seed → reproduzierbar).


In [190]:
# Zeitraster: ein Tag, 15-Minuten-Auflösung
dt_h = 0.25                                                  # Zeitschrittlänge [h]
zeit = pd.date_range("2019-07-23 00:00", "2019-07-23 23:45", freq="15min")
T    = len(zeit)                                             # Anzahl Zeitschritte (96)
stunde = zeit.hour + zeit.minute / 60                        # Uhrzeit als Dezimalzahl
rng  = np.random.default_rng(42)                             # reproduzierbares Rauschen

# --- Grundlast [MW]: Nachtsockel + Morgen-/Mittags-/kräftige Abendspitze + Welligkeit + Rauschen
grundlast = (26
             + 16 * np.exp(-((stunde -  8.5) ** 2) / (2 * 1.3 ** 2))   # Morgen
             + 20 * np.exp(-((stunde - 13.0) ** 2) / (2 * 2.2 ** 2))   # Mittagsplateau
             + 42 * np.exp(-((stunde - 19.0) ** 2) / (2 * 1.6 ** 2))   # kräftige Abendspitze
             +  4 * np.sin(stunde / 24 * 2 * np.pi * 3))               # Tageswelligkeit
grundlast = np.clip(grundlast + rng.normal(0, 1.6, T), 15, None)
grundlast = pd.Series(np.round(grundlast, 1), index=zeit, name="Grundlast")

# --- Wärmebedarf [MW_th]: Prozesswärme mit kräftiger Mittagsspitze + Morgen-/Abendspitze
q_bedarf = (12
            + 38 * np.exp(-((stunde -  6.5) ** 2) / (2 * 1.1 ** 2))    # Morgen-Aufheizen
            + 96 * np.exp(-((stunde - 12.5) ** 2) / (2 * 1.7 ** 2))    # Mittags-Prozessspitze
            + 62 * np.exp(-((stunde - 19.0) ** 2) / (2 * 1.7 ** 2))    # Abend-Prozesswärme
            +  6 * np.sin(stunde / 24 * 2 * np.pi * 5))
q_bedarf = np.clip(q_bedarf + rng.normal(0, 2.2, T), 8, None)
q_bedarf = pd.Series(np.round(q_bedarf, 1), index=zeit, name="Waermebedarf")

### 3 · Konfiguration der neuen Module & Netzgrenzen

Hier werden alle Vorgaben eingetragen (**Dummy-Werte, beliebig anpassbar**):
- **Freigabe-Stunden**, zu denen die Netzanschlussleistung überschritten werden darf
- **Leistung des Elektrodenkessels** (das Wärmelastprofil steht in Zelle 2)
- **Batteriespeicher-Kennwerte** (die Kapazität selbst wird von der Optimierung bestimmt)


In [191]:
# ==================================================================
# KONFIGURATION  –  hier die neuen Module & Netzgrenzen eintragen
# ==================================================================

# --- Netzanschluss ------------------------------------------------
P_netz_normal = 80.0      # reguläre Netzanschlussleistung [MW] (außerhalb Freigabe nicht überschreitbar)
P_netz_spitze = 130.0     # zulässige Leistung WÄHREND der Freigabe-Stunden [MW]
erlaubte_stunden = [5, 6, 11, 12, 18, 19, 20, 21, 22, 23, 24, 1, 2, 3, 4]   # Uhrzeiten [Std], zu denen überschritten werden darf

# --- Elektrodenkessel --------------------------------------------
P_ek_max = 50.0           # max. elektrische Leistung des Elektrodenkessels [MW_el]
eta_ek   = 0.99           # Wirkungsgrad Strom -> Wärme [-]

# --- Batteriespeicher (Kapazität E_bat wird optimiert) -----------
c_rate  = 0.5             # zul. Lade-/Entladeleistung als Anteil der Kapazität [1/h]  -> P_max = c_rate * E_bat
eta_bat = 0.95            # Lade-/Entladewirkungsgrad je Richtung [-]

# ------------------------------------------------------------------
# abgeleitete Zeitreihen
# ------------------------------------------------------------------
ist_freigabe = pd.Series(zeit.hour.isin(erlaubte_stunden), index=zeit)                 # bool je Zeitschritt
P_netz_limit = pd.Series(np.where(ist_freigabe, P_netz_spitze, P_netz_normal),
                         index=zeit, name="Netzlimit")

# elektrische Leistungsaufnahme des Kessels (durch Wärmeprofil vorgegeben, auf P_ek_max begrenzt)
p_ek = np.minimum(q_bedarf.to_numpy() / eta_ek, P_ek_max)
p_ek = pd.Series(np.round(p_ek, 2), index=zeit, name="Elektrodenkessel")

if np.any(q_bedarf.to_numpy() / eta_ek > P_ek_max + 1e-6):
    print("Hinweis: P_ek_max begrenzt das Wärmeprofil – Bedarf wird zeitweise gekappt.")

Hinweis: P_ek_max begrenzt das Wärmeprofil – Bedarf wird zeitweise gekappt.


### 4 · Ausgangssituation (vor Optimierung)

Grundlast und die zusätzliche elektrische Last des Elektrodenkessels. Die gestrichelte Linie
„Summe ohne Speicher" zeigt, was der Netzanschluss ohne Gegenmaßnahme sehen würde – deutlich über
der Netzanschlussleistung. Freigabe-Stunden orange hinterlegt.


In [192]:
def markiere_freigabe(fig, **kw):
    """Freigabe-Stunden im Plot orange hinterlegen."""
    tag0 = zeit[0].normalize()
    for h in sorted(set(erlaubte_stunden)):
        fig.add_vrect(x0=tag0 + pd.Timedelta(hours=h), x1=tag0 + pd.Timedelta(hours=h + 1),
                      fillcolor="LightSalmon", opacity=0.13, line_width=0, layer="below", **kw)

netz_ohne_speicher = (grundlast + p_ek).rename("Summe ohne Speicher")

fig = go.Figure()
fig.add_trace(go.Scatter(x=zeit, y=grundlast, name="Grundlast (Bestand)",
                         line_shape="hv", stackgroup="v", line=dict(width=0, color=FARBEN["grund"])))
fig.add_trace(go.Scatter(x=zeit, y=p_ek, name="Elektrodenkessel (elektr.)",
                         line_shape="hv", stackgroup="v", line=dict(width=0, color=FARBEN["kessel"])))
fig.add_trace(go.Scatter(x=zeit, y=netz_ohne_speicher, name="Summe ohne Speicher",
                         line_shape="hv", line=dict(color=FARBEN["netz"], width=2, dash="dot")))
fig.add_trace(go.Scatter(x=zeit, y=[P_netz_normal] * T, name="Netzanschlussleistung",
                         line=dict(color=FARBEN["limit"], dash="dash", width=2)))
markiere_freigabe(fig)
basis_layout(fig, "Ausgangssituation: Grundlast + Elektrodenkessel vs. Netzanschluss")
fig.show(config=PLOT_CONFIG)

### 5 · Optimierungsmodell (Pyomo)

**Entscheidungsvariablen** je Zeitschritt $t$: Netzbezug $p_{grid}$, Batterie laden $p_{ch}$,
Batterie entladen $p_{dis}$, Ladestand $soc$; sowie die **Batteriekapazität** $E_{bat}$ (skalar).
Die Leistungsaufnahme des Elektrodenkessels $p_{ek}(t)$ ist durch das Wärmeprofil vorgegeben.

| | Bezeichnung | Gleichung |
|---|---|---|
| I  | Leistungsbilanz | $p_{grid}(t) + p_{dis}(t) = p_{grund}(t) + p_{ek}(t) + p_{ch}(t)\quad\forall t$ |
| II | Netzanschluss-Grenze | $p_{grid}(t) \le P_{limit}(t)\quad\forall t$  (in Freigabe-Std. höher) |
| III | SOC-Dynamik | $soc(t) = soc(t{-}1) + \big(p_{ch}(t)\,\eta - p_{dis}(t)/\eta\big)\,\Delta t$ |
| IV | Kapazitätsgrenze | $soc(t) \le E_{bat}\quad\forall t$ |
| V  | Leistungsgrenze Batterie | $p_{ch}(t),\,p_{dis}(t) \le c_{rate}\cdot E_{bat}\quad\forall t$ |
| VI | Tageszyklus | $soc(T{-}1) = soc_{init}$ |

**Zielfunktion:** $\min\; E_{bat} + \varepsilon\sum_t p_{grid}(t)\,\Delta t$
(minimale Batteriegröße; der kleine Term vermeidet unnötiges Speicherzyklen).


In [193]:
m = pyoe.ConcreteModel("Lastspitzenmanagement")
m.T = pyoe.Set(initialize=range(T))

base  = grundlast.to_numpy()
ekel  = p_ek.to_numpy()
limit = P_netz_limit.to_numpy()

# --- Variablen ----------------------------------------------------
m.E_bat    = pyoe.Var(bounds=(0, None))              # Batteriekapazität [MWh]  (zu dimensionieren)
m.soc_init = pyoe.Var(bounds=(0, None))              # Anfangsladestand [MWh]
m.p_grid   = pyoe.Var(m.T, bounds=(0, None))         # Netzbezug [MW]
m.p_ch     = pyoe.Var(m.T, bounds=(0, None))         # Batterie laden [MW]
m.p_dis    = pyoe.Var(m.T, bounds=(0, None))         # Batterie entladen [MW]
m.soc      = pyoe.Var(m.T, bounds=(0, None))         # Ladestand [MWh]

# --- Nebenbedingungen ---------------------------------------------
m.bilanz  = pyoe.ConstraintList()
m.netz    = pyoe.ConstraintList()
m.socdyn  = pyoe.ConstraintList()
m.soccap  = pyoe.ConstraintList()
m.p_ch_l  = pyoe.ConstraintList()
m.p_dis_l = pyoe.ConstraintList()

for t in m.T:
    m.bilanz.add(m.p_grid[t] + m.p_dis[t] == base[t] + ekel[t] + m.p_ch[t])          # I
    m.netz.add(m.p_grid[t] <= limit[t])                                              # II
    if t == 0:                                                                        # III
        m.socdyn.add(m.soc[t] == m.soc_init + (m.p_ch[t] * eta_bat - m.p_dis[t] / eta_bat) * dt_h)
    else:
        m.socdyn.add(m.soc[t] == m.soc[t - 1] + (m.p_ch[t] * eta_bat - m.p_dis[t] / eta_bat) * dt_h)
    m.soccap.add(m.soc[t] <= m.E_bat)                                                 # IV
    m.p_ch_l.add(m.p_ch[t]  <= c_rate * m.E_bat)                                      # V
    m.p_dis_l.add(m.p_dis[t] <= c_rate * m.E_bat)

m.soc_init_cap = pyoe.Constraint(expr=m.soc_init <= m.E_bat)
m.zyklus = pyoe.Constraint(expr=m.soc[T - 1] == m.soc_init)                           # VI

eps = 1e-4
m.obj = pyoe.Objective(expr=m.E_bat + eps * pyoe.quicksum(m.p_grid[t] * dt_h for t in m.T),
                       sense=pyoe.minimize)

### 6 · Solver aufrufen

In [194]:
solver = SolverFactory("gurobi")     # LP-Modell; alternativ "cbc" / "glpk"
results = solver.solve(m)
print("Solver-Status :", results.solver.status)
print("Abbruchgrund  :", results.solver.termination_condition)
print(f"Dimensionierte Batteriekapazität: {pyoe.value(m.E_bat):.1f} MWh "
      f"(max. Leistung {c_rate * pyoe.value(m.E_bat):.1f} MW)")

Solver-Status : ok
Abbruchgrund  : optimal
Dimensionierte Batteriekapazität: 62.0 MWh (max. Leistung 31.0 MW)


### 7 · Ergebnisse aufbereiten

In [195]:
res = pd.DataFrame(index=zeit)
res["Grundlast"]        = base
res["Elektrodenkessel"] = ekel
res["Bat_laden"]        = [pyoe.value(m.p_ch[t])  for t in m.T]
res["Bat_entladen"]     = [pyoe.value(m.p_dis[t]) for t in m.T]
res["Netzbezug"]        = [pyoe.value(m.p_grid[t]) for t in m.T]
res["SOC"]              = [pyoe.value(m.soc[t])   for t in m.T]
res["Netzlimit"]        = limit
res["Gesamtlast"]       = res["Grundlast"] + res["Elektrodenkessel"] + res["Bat_laden"]  # Verbrauchsseite

E_bat_opt = pyoe.value(m.E_bat)
P_bat_opt = c_rate * E_bat_opt

### 8 · Hauptplot: Optimierter Betrieb

Verbrauchsseite als gestapelte Flächen (Grundlast + Kessel + Batterieladung = Gesamtlast).
Die **rote gestrichelte Linie** ist die reguläre Netzanschlussleistung – sichtbar, wo sie
überschritten wird. Die **schwarze Linie** ist der tatsächliche Netzbezug; er bleibt außerhalb der
Freigabe-Stunden (orange) darunter, weil die **Batterie** die Differenz liefert.


In [196]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32],
                    vertical_spacing=0.09,
                    specs=[[{"secondary_y": False}], [{"secondary_y": True}]])

# oben: gestapelte Verbrauchsflächen
fig.add_trace(go.Scatter(x=res.index, y=res["Grundlast"], name="Grundlast", line_shape="hv",
                         stackgroup="v", line=dict(width=0, color=FARBEN["grund"])), row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=res["Elektrodenkessel"], name="Elektrodenkessel",
                         line_shape="hv", stackgroup="v", line=dict(width=0, color=FARBEN["kessel"])), row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=res["Bat_laden"], name="Batterie laden", line_shape="hv",
                         stackgroup="v", line=dict(width=0, color=FARBEN["laden"])), row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=res["Netzbezug"], name="Netzbezug", line_shape="hv",
                         line=dict(color=FARBEN["netz"], width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=[P_netz_normal] * T, name="Netzanschlussleistung",
                         line=dict(color=FARBEN["limit"], dash="dash", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=res["Netzlimit"], name="zul. Grenze (mit Freigabe)",
                         line_shape="hv", line=dict(color="darkorange", dash="dot", width=1.5)), row=1, col=1)

# unten: Batterie
fig.add_trace(go.Scatter(x=res.index, y=res["Bat_laden"], name="laden", line_shape="hv",
                         fill="tozeroy", line=dict(color=FARBEN["laden"])), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=res.index, y=-res["Bat_entladen"], name="entladen", line_shape="hv",
                         fill="tozeroy", line=dict(color=FARBEN["entladen"])), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=res.index, y=res["SOC"], name="Ladestand (SOC)", line_shape="hv",
                         line=dict(color=FARBEN["soc"], width=2)), row=2, col=1, secondary_y=True)

markiere_freigabe(fig)
fig.update_yaxes(title_text="Leistung [MW]", row=1, col=1)
fig.update_yaxes(title_text="Batterie [MW]", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="SOC [MWh]", row=2, col=1, secondary_y=True)
fig.update_xaxes(tickformat="%H:%M", dtick=3 * 3600 * 1000)   # Zeitachse technisch: nur HH:MM
fig.update_layout(title=dict(text=f"Optimierter Betrieb – dimensionierte Batterie: {E_bat_opt:.0f} MWh / {P_bat_opt:.0f} MW",
                             x=0.0, xanchor="left", y=0.985, yanchor="top"),
                  height=780, width=1000,
                  legend=dict(orientation="h", yanchor="top", y=-0.12, x=0),
                  margin=dict(t=80, r=30, l=75, b=115))
fig.show(config=PLOT_CONFIG)

### 9 · Peak-Shaving: Netzbezug mit vs. ohne Batterie

„Ohne Batterie" entspricht Grundlast + Kessel (der Netzanschluss würde die volle Spitze sehen).
Innerhalb der Freigabe-Stunden (orange) darf der Netzbezug bis zur erhöhten Grenze steigen –
**außerhalb** der Freigabe kappt die Batterie den Netzbezug auf die reguläre Anschlussleistung.


In [197]:
# außerhalb der Freigabe-Stunden: dort muss die Batterie die Spitze kappen
maske_aus = ~ist_freigabe.to_numpy()
peak_ohne_aus = netz_ohne_speicher.to_numpy()[maske_aus].max()
idx_ohne_aus  = np.where(maske_aus & (netz_ohne_speicher.to_numpy() == peak_ohne_aus))[0][0]
peak_mit_aus  = res["Netzbezug"].to_numpy()[maske_aus].max()

fig = go.Figure()
fig.add_trace(go.Scatter(x=zeit, y=netz_ohne_speicher, name="Netzbezug OHNE Batterie",
                         line_shape="hv", fill="tozeroy", line=dict(color="#bbbbbb", width=1.5)))
fig.add_trace(go.Scatter(x=res.index, y=res["Netzbezug"], name="Netzbezug MIT Batterie",
                         line_shape="hv", line=dict(color=FARBEN["netz"], width=2.5)))
fig.add_trace(go.Scatter(x=zeit, y=[P_netz_normal] * T, name="Netzanschlussleistung",
                         line=dict(color=FARBEN["limit"], dash="dash", width=2)))
fig.add_trace(go.Scatter(x=zeit, y=res["Netzlimit"], name="zul. Grenze (mit Freigabe)",
                         line_shape="hv", line=dict(color="darkorange", dash="dot", width=1.5)))
markiere_freigabe(fig)
basis_layout(fig, "Peak-Shaving der Batterie außerhalb der Freigabe-Stunden")
fig.add_annotation(x=zeit[idx_ohne_aus], y=peak_ohne_aus,
                   text=f"ohne Batterie: {peak_ohne_aus:.0f} MW<br>(unzulässig)", showarrow=True,
                   arrowhead=2, ay=-35, font=dict(color=FARBEN["limit"]))
fig.add_annotation(x=zeit[idx_ohne_aus], y=peak_mit_aus,
                   text=f"mit Batterie: {peak_mit_aus:.0f} MW", showarrow=True, arrowhead=2, ay=35)
fig.show(config=PLOT_CONFIG)

### 10 · Lastdauerlinie

Nach Höhe sortierte Leistung über den Tag: zeigt, wie viele Stunden welche Leistung ansteht.
Die Batterie schneidet den oberen Teil der Netzbezugs-Dauerlinie ab.


In [198]:
dauer = np.arange(1, T + 1) * dt_h    # kumulierte Stunden
gl_sort = np.sort(res["Gesamtlast"].to_numpy())[::-1]
nb_sort = np.sort(res["Netzbezug"].to_numpy())[::-1]
oh_sort = np.sort(netz_ohne_speicher.to_numpy())[::-1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=dauer, y=oh_sort, name="Netzbezug ohne Batterie", line_shape="hv",
                         line=dict(color="#bbbbbb", width=2)))
fig.add_trace(go.Scatter(x=dauer, y=nb_sort, name="Netzbezug mit Batterie", line_shape="hv",
                         line=dict(color=FARBEN["netz"], width=2.5)))
fig.add_trace(go.Scatter(x=dauer, y=[P_netz_normal] * T, name="Netzanschlussleistung",
                         line=dict(color=FARBEN["limit"], dash="dash", width=2)))
basis_layout(fig, "Lastdauerlinie des Netzbezugs", zeit_achse=False)
fig.update_xaxes(title_text="Dauer [h]")
fig.show(config=PLOT_CONFIG)

### 11 · Ladestand (SOC) des Batteriespeichers

Eigenständiger SOC-Verlauf über den Tag inkl. dimensionierter Kapazität.


In [199]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=res.index, y=res["SOC"], name="Ladestand", line_shape="hv",
                         fill="tozeroy", line=dict(color=FARBEN["soc"], width=2)))
fig.add_trace(go.Scatter(x=res.index, y=[E_bat_opt] * T, name="dimensionierte Kapazität",
                         line=dict(color=FARBEN["soc"], dash="dash", width=1.5)))
markiere_freigabe(fig)
basis_layout(fig, f"Ladestand des Batteriespeichers (Kapazität {E_bat_opt:.0f} MWh)", ytitel="SOC [MWh]")
fig.show(config=PLOT_CONFIG)

### 12 · Tages-Energiebilanz

Umgesetzte Energiemengen über den Tag: Verbrauch (Grundlast, Kessel), Netzbezug, Batteriehub und
Speicherverluste.


In [200]:
e = {
    "Grundlast":        res["Grundlast"].sum() * dt_h,
    "Elektrodenkessel": res["Elektrodenkessel"].sum() * dt_h,
    "Netzbezug":        res["Netzbezug"].sum() * dt_h,
    "Batterie laden":   res["Bat_laden"].sum() * dt_h,
    "Batterie entladen":res["Bat_entladen"].sum() * dt_h,
}
e["Speicherverluste"] = e["Batterie laden"] - e["Batterie entladen"]

bar_farben = [FARBEN["grund"], FARBEN["kessel"], FARBEN["netz"], FARBEN["laden"], FARBEN["entladen"], "#999999"]
fig = go.Figure(go.Bar(x=list(e.keys()), y=[round(v, 1) for v in e.values()],
                       text=[f"{v:.0f}" for v in e.values()], textposition="outside",
                       marker_color=bar_farben))
basis_layout(fig, "Tages-Energiebilanz", ytitel="Energie [MWh]", zeit_achse=False)
fig.update_xaxes(title_text="")
fig.show(config=PLOT_CONFIG)

### 13 · Kennzahlen

In [201]:
maske_aus     = ~ist_freigabe.to_numpy()
peak_ohne_aus = netz_ohne_speicher.to_numpy()[maske_aus].max()   # Spitze außerhalb Freigabe ohne Batterie
peak_mit_aus  = res["Netzbezug"].to_numpy()[maske_aus].max()     # ... mit Batterie
peak_freigabe = res["Netzbezug"].to_numpy()[~maske_aus].max()    # Netzbezugsspitze in Freigabe-Stunden
ueber_std     = (res["Gesamtlast"] > P_netz_normal + 1e-6).sum() * dt_h
netz_ueber    = (res["Netzbezug"] > P_netz_normal + 1e-6).sum() * dt_h
e_netz        = res["Netzbezug"].sum() * dt_h

print(f"Dimensionierte Batteriekapazität         : {E_bat_opt:8.1f} MWh")
print(f"  -> zul. Lade-/Entladeleistung           : {P_bat_opt:8.1f} MW")
print(f"reguläre Netzanschlussleistung           : {P_netz_normal:8.1f} MW  (Freigabe bis {P_netz_spitze:.0f} MW)")
print(f"Netzbezugsspitze IN  Freigabe-Stunden    : {peak_freigabe:8.1f} MW  (zulässige Überschreitung)")
print(f"Spitze AUSSERHALB Freigabe ohne Batterie : {peak_ohne_aus:8.1f} MW  (wäre unzulässig)")
print(f"Spitze AUSSERHALB Freigabe mit  Batterie : {peak_mit_aus:8.1f} MW")
print(f"  -> durch Batterie gekappt (Peak-Shaving): {peak_ohne_aus - peak_mit_aus:8.1f} MW")
print(f"Dauer Gesamtlast > Anschlussleistung     : {ueber_std:8.2f} h  (per Batterie/Freigabe abgefangen)")
print(f"Dauer Netzbezug  > Anschlussleistung     : {netz_ueber:8.2f} h  (nur in Freigabe-Stunden)")
print(f"Netzbezug gesamt (Tag)                   : {e_netz:8.1f} MWh")

Dimensionierte Batteriekapazität         :     62.0 MWh
  -> zul. Lade-/Entladeleistung           :     31.0 MW
reguläre Netzanschlussleistung           :     80.0 MW  (Freigabe bis 130 MW)
Netzbezugsspitze IN  Freigabe-Stunden    :    126.6 MW  (zulässige Überschreitung)
Spitze AUSSERHALB Freigabe ohne Batterie :    111.0 MW  (wäre unzulässig)
Spitze AUSSERHALB Freigabe mit  Batterie :     80.0 MW
  -> durch Batterie gekappt (Peak-Shaving):     31.0 MW
Dauer Gesamtlast > Anschlussleistung     :    12.75 h  (per Batterie/Freigabe abgefangen)
Dauer Netzbezug  > Anschlussleistung     :     6.00 h  (nur in Freigabe-Stunden)
Netzbezug gesamt (Tag)                   :   1809.7 MWh


### 14 · Gegenüberstellung: Aktueller Stand vs. neuer Verlauf

**Oben – aktueller Stand (ohne Maßnahme):** Grundlast + Elektrodenkessel und der resultierende
Netzbezug. Der Anteil **über** der Netzanschlussleistung (die Überschreitung) ist **rot** dargestellt.

**Unten – neuer Verlauf (mit Batteriespeicher & Freigabe):** resultierende Last (Netzbezug) sowie
Beladung und Entladung des Speichers. Zeitfenster, in denen die Anschlussleistung **erhöht** werden
darf, sind dezent **grün** hinterlegt, die übrigen **rot**. Der Batteriespeicher ist in gedeckten
Stahlblau-Tönen dargestellt (hell = Beladung, dunkel = Entladung).


In [ ]:
# ---- Datenaufbereitung -------------------------------------------------
grund     = res["Grundlast"].to_numpy()
kessel    = res["Elektrodenkessel"].to_numpy()
total_akt = grund + kessel                 # aktueller Netzbezug OHNE Batterie
lim       = P_netz_normal                   # Bezugsgröße für die Überschreitung (rot)

# Aufteilung oberer Plot: unterhalb Grenze gedeckt-blau, oberhalb rot (Überschreitung)
unter_grund  = np.minimum(grund, lim)
unter_kessel = np.clip(np.minimum(total_akt, lim) - grund, 0, None)
ueberschr    = np.clip(total_akt - lim, 0, None)

# gedeckte, technische Palette (bewusst zurückhaltend)
C_GRUND  = "#cdd6df"     # Grundlast        (helles Kühlgrau-Blau)
C_KESSEL = "#4f6981"     # Elektrodenkessel (gedecktes Stahlblau)
C_UEBER  = "rgba(150,52,52,0.72)"   # Überschreitung (gedecktes Ziegelrot)
C_NETZ   = "#1a1a1a"     # Netzbezug / resultierende Last
C_LIMIT  = "#7a2222"     # Netzanschlussleistung (dunkelrot, gestrichelt)
C_LADEN  = "#7d97ad"     # Beladung  (helles Stahlblau)
C_ENTL   = "#2f4557"     # Entladung (dunkles Stahlblau)
BG_GRUEN = "rgba(96,128,96,0.1)"   # Freigabe-Fenster (entsättigt)
BG_ROT   = "rgba(150,92,92,0.8)"   # kein Freigabe-Fenster (entsättigt)

nfig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5],
                     vertical_spacing=0.12,
                     subplot_titles=("Aktueller Stand (ohne Maßnahme)",
                                     "Neuer Verlauf (mit Batteriespeicher & Freigabe)"))

# ===================== oben: aktueller Stand =====================
nfig.add_trace(go.Scatter(x=zeit, y=unter_grund, name="Grundlast", line_shape="hv",
                          stackgroup="akt", line=dict(width=0), fillcolor=C_GRUND), row=1, col=1)
nfig.add_trace(go.Scatter(x=zeit, y=unter_kessel, name="Elektrodenkessel", line_shape="hv",
                          stackgroup="akt", line=dict(width=0), fillcolor=C_KESSEL), row=1, col=1)
nfig.add_trace(go.Scatter(x=zeit, y=ueberschr, name="Überschreitung", line_shape="hv",
                          stackgroup="akt", line=dict(width=0), fillcolor=C_UEBER), row=1, col=1)
nfig.add_trace(go.Scatter(x=zeit, y=total_akt, name="Netzbezug (aktuell)", line_shape="hv",
                          line=dict(color=C_NETZ, width=1.8)), row=1, col=1)
nfig.add_trace(go.Scatter(x=zeit, y=[lim] * T, name="Netzanschlussleistung",
                          line=dict(color=C_LIMIT, dash="dash", width=1.8)), row=1, col=1)

# ===================== unten: neuer Verlauf =====================
# Hintergrundfelder: Freigabe -> grün (Erhöhung möglich), sonst -> rot (dezent)
tag0 = zeit[0].normalize()
freigabe_set = set(erlaubte_stunden)
for h in range(24):
    x0 = tag0 + pd.Timedelta(hours=h); x1 = x0 + pd.Timedelta(hours=1)
    farbe = BG_GRUEN if h in freigabe_set else BG_ROT
    nfig.add_shape(type="rect", xref="x2", yref="y2 domain", x0=x0, x1=x1, y0=0, y1=1,
                   fillcolor=farbe, line_width=0, layer="below")

nfig.add_trace(go.Scatter(x=res.index, y=res["Bat_laden"], name="Beladung", line_shape="hv",
                          fill="tozeroy", line=dict(color=C_LADEN, width=1.2),
                          fillcolor="rgba(125,151,173,0.55)"), row=2, col=1)
nfig.add_trace(go.Scatter(x=res.index, y=-res["Bat_entladen"], name="Entladung", line_shape="hv",
                          fill="tozeroy", line=dict(color=C_ENTL, width=1.2),
                          fillcolor="rgba(47,69,87,0.50)"), row=2, col=1)
nfig.add_trace(go.Scatter(x=res.index, y=res["Netzbezug"], name="resultierende Last (Netzbezug)",
                          line_shape="hv", line=dict(color=C_NETZ, width=2.2)), row=2, col=1)
nfig.add_trace(go.Scatter(x=res.index, y=[P_netz_normal] * T, name="Netzanschlussleistung",
                          line=dict(color=C_LIMIT, dash="dash", width=1.8), showlegend=False), row=2, col=1)

# Legenden-Platzhalter für die grün/rot hinterlegten Felder
nfig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Erhöhung möglich (Freigabe)",
                          marker=dict(size=13, symbol="square", color="rgba(96,128,96,0.55)")), row=2, col=1)
nfig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="keine Erhöhung",
                          marker=dict(size=13, symbol="square", color="rgba(150,92,92,0.55)")), row=2, col=1)

nfig.update_yaxes(title_text="Leistung [MW]", row=1, col=1)
nfig.update_yaxes(title_text="Leistung [MW]", row=2, col=1)
nfig.update_xaxes(tickformat="%H:%M", dtick=3 * 3600 * 1000)
nfig.update_layout(height=820, width=1000,
                   legend=dict(orientation="h", yanchor="top", y=-0.10, x=0),
                   margin=dict(t=70, r=30, l=75, b=130))
nfig.show(config=PLOT_CONFIG)

: 